In [ ]:
# ============================================================
# Publication-grade 2D flattened baselines for PMA prediction
#
# Input:
#   Same 120 min window + 40 min stride, imputed 2-channel CTHW
#   dataset used by the CNN main analysis.
#
# Models:
#   1) Ridge regression (linear baseline)
#   2) XGBoost regression (nonlinear tree baseline)
#
# Experimental protocol:
#   - Reuse the exact outer-test patient folds saved by the main CNN run.
#   - Recreate the same 3 inner patient folds inside each outer-train fold.
#   - Optuna tunes model hyperparameters only; there is no architecture search.
#   - Training-fold-only, patient-balanced normalization.
#   - Patient-balanced sample weights: each patient has total weight 1.
#   - Model selection uses mean inner-validation patient-level MAE.
#   - Final model is refit on all outer-train patients and evaluated once on
#     outer-test patients.
#   - Ridge is deterministic and is refit once. XGBoost uses a 3-seed final
#     ensemble to mirror the stochastic robustness of the CNN main analysis.
#   - Complete OOF window-level and patient-level predictions are saved.
#
# 2D flattening:
#   [normalized value channel (T x F_dynamic),
#    unchanged mask channel      (T x F_dynamic),
#    normalized static features]
#   -> one flat vector per 120 min window.
#
# Important:
#   This script assumes the upstream PKL is already imputed exactly as in the
#   CNN main analysis and contains no repeated/current-weight feature.
# ============================================================

from __future__ import annotations

import gc
import glob
import hashlib
import json
import math
import os
import random
import shutil
import sys
import warnings
from collections import Counter, OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold

try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "xgboost is required. Install it in the active environment, e.g. "
        "`pip install xgboost`."
    ) from exc

warnings.filterwarnings("default")


# ============================================================
# 0. Configuration
# ============================================================
SEED = 42
PROJECT_ROOT = "/mnt/home/qinqiu/thesis"

DATASET_ROOT = os.path.join(
    PROJECT_ROOT,
    "data",
    "superwindow_stride_comparison_90_120_3h_no_repeated_weight_no_los_nec_ga_le_35_no_intubation_imputed_2ch_CTHW",
)
DATASET_LABEL = "120min_stride40min_overlap67"

# Set this to an exact PKL path to bypass automatic search.
DATA_PKL_OVERRIDE: Optional[str] = None

# Completed main CNN experiment. The baseline reads its saved outer-test fold
# assignment so that every patient's test fold is identical across models.
MAIN_CNN_EXP_ROOT = os.path.join(
    PROJECT_ROOT,
    "result_nested",
    "nested_cv_120min_stride40min_v7_dataset_reuse_patientbalanced_resume100",
)
MAIN_OUTER_ASSIGNMENT_CSV = os.path.join(
    MAIN_CNN_EXP_ROOT,
    "outer_fold_assignment_long.csv",
)
MAIN_CNN_PATIENT_OOF_CSV = os.path.join(
    MAIN_CNN_EXP_ROOT,
    "FINAL_nested_oof_patient_predictions.csv",
)

EXP_ROOT = os.path.join(
    PROJECT_ROOT,
    "result_baseline",
    "baseline_2d_flattened_ridge_xgboost_nestedcv_optuna",
)
os.makedirs(EXP_ROOT, exist_ok=True)

OUTER_SPLITS = 5
INNER_SPLITS = 3

# Ridge has essentially one important complexity parameter, so 30 trials are
# already generous. XGBoost has a substantially larger search space.
N_TRIALS_PER_OUTER: Dict[str, int] = {
    "ridge": 30,
    "xgboost": 100,
}

MODELS_TO_RUN: Tuple[str, ...] = ("ridge", "xgboost")

# Ridge is deterministic. Repeating it with three seeds gives identical output.
# XGBoost is stochastic and therefore uses the same three final seeds as CNN.
FINAL_REFIT_SEEDS: Dict[str, List[int]] = {
    "ridge": [42],
    "xgboost": [42, 123, 2026],
}

# XGBoost configuration. For xgboost >= 2, device="cuda" with tree_method="hist"
# is used. For xgboost < 2, tree_method="gpu_hist" is used instead.
# Change to "cpu" if GPU XGBoost is unavailable in the environment.
XGB_DEVICE = "cuda"  # "cuda" or "cpu"
XGB_N_JOBS = 4

STATIC_FEATURES = ["feats__ga_w", "feats__sex", "feats__bw"]
FORBIDDEN_FEATURES = ["feats__weight"]
STRICT_NO_REPEATED_WEIGHT = True

INCLUDE_MASK_CHANNEL = True
PATIENT_BALANCED_NORMALIZATION = True
PATIENT_BALANCED_TRAINING = True
REQUIRE_MAIN_CNN_OUTER_FOLDS = True

# Prepared split matrices are persisted as .npy files and loaded with mmap.
# This avoids rebuilding normalization and flattening for every Optuna trial.
CACHE_PREPARED_SPLITS = True

# Set smaller values for a quick code smoke test, then restore for formal runs.
SMOKE_TEST = False
if SMOKE_TEST:
    N_TRIALS_PER_OUTER = {"ridge": 3, "xgboost": 3}

CODE_VERSION = "baseline_2d_flattened_nestedcv_v1"
NORMALIZATION_MODE = (
    "patient_balanced_per_dynamic_feature_zscore_training_fold_only_"
    "static_zscore_training_fold_only_mask_unchanged"
)


# ============================================================
# 1. Reproducibility and atomic I/O
# ============================================================
def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def cleanup() -> None:
    gc.collect()


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (Path,)):
        return str(value)
    return value


def stable_hash(payload: Any) -> str:
    encoded = json.dumps(
        json_safe(payload),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def atomic_json_dump(payload: Any, path: str) -> None:
    path = os.path.abspath(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(json_safe(payload), f, indent=2, sort_keys=True)
    os.replace(tmp, path)


def atomic_npy_save(array: np.ndarray, path: str) -> None:
    path = os.path.abspath(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp.npy"
    np.save(tmp, array)
    os.replace(tmp, path)


def atomic_joblib_dump(obj: Any, path: str) -> None:
    path = os.path.abspath(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    joblib.dump(obj, tmp)
    os.replace(tmp, path)


def read_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


seed_everything(SEED)


# ============================================================
# 2. Find and load the same 120 min dataset used by CNN
# ============================================================
def find_dataset_file(dataset_root: str, label: str) -> str:
    patterns = [
        os.path.join(dataset_root, f"*{label}*imputed*2ch*CTHW*.pkl"),
        os.path.join(dataset_root, f"*{label}*2ch*CTHW*.pkl"),
        os.path.join(dataset_root, f"*{label}*imputed*.pkl"),
        os.path.join(dataset_root, f"*{label}*.pkl"),
    ]

    matches: List[str] = []
    for pattern in patterns:
        matches.extend(glob.glob(pattern))
    matches = list(OrderedDict.fromkeys(matches))

    if not matches:
        raise FileNotFoundError(
            f"Could not find a dataset for label={label} under {dataset_root}. "
            "Set DATA_PKL_OVERRIDE to the exact file."
        )

    def score_path(path: str) -> int:
        name = os.path.basename(path).lower()
        score = 0
        score += 10 if "imputed" in name else 0
        score += 10 if "2ch" in name else 0
        score += 10 if "cthw" in name else 0
        score += 3 if "loose80" in name else 0
        score += 20 if "no_repeated_weight" in path.lower() else 0
        return score

    scored = [(score_path(p), os.path.abspath(p)) for p in matches]
    best_score = max(score for score, _ in scored)
    best = sorted(path for score, path in scored if score == best_score)
    if len(best) != 1:
        raise RuntimeError(
            "Dataset auto-selection is ambiguous. Set DATA_PKL_OVERRIDE. "
            f"Top-scoring matches: {best}"
        )
    return best[0]


if DATA_PKL_OVERRIDE is not None:
    DATA_PKL = os.path.abspath(DATA_PKL_OVERRIDE)
    if not os.path.isfile(DATA_PKL):
        raise FileNotFoundError(f"DATA_PKL_OVERRIDE does not exist: {DATA_PKL}")
else:
    DATA_PKL = find_dataset_file(DATASET_ROOT, DATASET_LABEL)

print("DATA_PKL:", DATA_PKL)
print("MAIN_OUTER_ASSIGNMENT_CSV:", MAIN_OUTER_ASSIGNMENT_CSV)
print("EXP_ROOT:", EXP_ROOT)
print("XGBoost version:", xgb.__version__)
print("XGB_DEVICE:", XGB_DEVICE)

with open(DATA_PKL, "rb") as f:
    data = joblib.load(f) if DATA_PKL.endswith(".joblib") else __import__("pickle").load(f)

required_keys = {"X2_by_patient", "y_by_patient", "feature_cols"}
missing_keys = required_keys - set(data.keys())
if missing_keys:
    raise KeyError(f"PKL is missing keys: {sorted(missing_keys)}")

X2_by_patient_raw: Mapping[Any, np.ndarray] = data["X2_by_patient"]
y_by_patient: Mapping[Any, np.ndarray] = data["y_by_patient"]
feature_cols = list(data["feature_cols"])

missing_static = [c for c in STATIC_FEATURES if c not in feature_cols]
if missing_static:
    raise ValueError(f"Missing static features: {missing_static}")

found_forbidden = [c for c in FORBIDDEN_FEATURES if c in feature_cols]
if STRICT_NO_REPEATED_WEIGHT and found_forbidden:
    raise ValueError(
        "Forbidden repeated/current-weight feature found: "
        f"{found_forbidden}. Use the no_repeated_weight PKL."
    )

static_idx = [feature_cols.index(c) for c in STATIC_FEATURES]
dynamic_idx = [i for i, c in enumerate(feature_cols) if c not in STATIC_FEATURES]
static_feature_cols = [feature_cols[i] for i in static_idx]
dynamic_feature_cols = [feature_cols[i] for i in dynamic_idx]

if not dynamic_feature_cols:
    raise ValueError("No dynamic features remain after separating static features.")

example_pid = next(
    (
        pid
        for pid, Xp in X2_by_patient_raw.items()
        if Xp is not None and hasattr(Xp, "shape") and Xp.shape[0] > 0
    ),
    None,
)
if example_pid is None:
    raise ValueError("No valid patient array found.")

example_shape = tuple(np.asarray(X2_by_patient_raw[example_pid]).shape)
if len(example_shape) != 4 or example_shape[1] != 2:
    raise ValueError(
        "Expected X shape (n_windows, 2, T, F), "
        f"got {example_shape}"
    )
if example_shape[-1] != len(feature_cols):
    raise ValueError("feature_cols length does not match X feature dimension.")

N_TIME = int(example_shape[2])
N_DYNAMIC_FEATURES = len(dynamic_feature_cols)
N_STATIC = len(static_feature_cols)

# Split dynamic and static arrays once, using np.take to avoid the NumPy
# advanced-indexing axis-order bug.
X2_by_patient: Dict[Any, Optional[np.ndarray]] = {}
Xstatic_by_patient: Dict[Any, Optional[np.ndarray]] = {}

for pid, Xp_raw in X2_by_patient_raw.items():
    if Xp_raw is None:
        X2_by_patient[pid] = None
        Xstatic_by_patient[pid] = None
        continue

    Xp = np.asarray(Xp_raw)
    if Xp.ndim != 4:
        raise ValueError(f"pid={pid}: expected 4D X, got {Xp.shape}")
    if Xp.shape[1:] != (2, N_TIME, len(feature_cols)):
        raise ValueError(
            f"pid={pid}: inconsistent X shape {Xp.shape}; expected "
            f"(n_windows, 2, {N_TIME}, {len(feature_cols)})"
        )
    if not np.isfinite(Xp).all():
        raise ValueError(
            f"pid={pid}: non-finite values remain in imputed X "
            f"({int((~np.isfinite(Xp)).sum())})"
        )

    if Xp.shape[0] == 0:
        X2_by_patient[pid] = np.empty(
            (0, 2, N_TIME, N_DYNAMIC_FEATURES), dtype=np.float32
        )
        Xstatic_by_patient[pid] = np.empty((0, N_STATIC), dtype=np.float32)
        continue

    static_all = np.take(Xp[:, 0, :, :], static_idx, axis=-1)
    static_reference = np.take(Xp[0, 0, 0, :], static_idx, axis=-1)

    if static_all.shape != (Xp.shape[0], N_TIME, N_STATIC):
        raise RuntimeError(
            f"pid={pid}: static indexing shape {static_all.shape} is invalid."
        )
    if not np.allclose(
        static_all,
        static_reference.reshape(1, 1, -1),
        rtol=0.0,
        atol=1e-6,
        equal_nan=False,
    ):
        raise ValueError(
            f"pid={pid}: STATIC_FEATURES vary across windows/time."
        )

    X2_by_patient[pid] = Xp[:, :, :, dynamic_idx].astype(np.float32)
    Xstatic_by_patient[pid] = Xp[:, 0, 0, static_idx].astype(np.float32)

pd.DataFrame({"dynamic_feature": dynamic_feature_cols}).to_csv(
    os.path.join(EXP_ROOT, "dynamic_features_for_2d_baseline.csv"), index=False
)
pd.DataFrame({"static_feature": static_feature_cols}).to_csv(
    os.path.join(EXP_ROOT, "static_features_for_2d_baseline.csv"), index=False
)

print("Example patient:", example_pid)
print("Raw example shape:", example_shape)
print("N_TIME:", N_TIME)
print("N_DYNAMIC_FEATURES:", N_DYNAMIC_FEATURES)
print("N_STATIC:", N_STATIC)
print("Flattened input dimension:", 2 * N_TIME * N_DYNAMIC_FEATURES + N_STATIC)


# ============================================================
# 3. Patient validation and summary
# ============================================================
def patient_invalid_reason(pid: Any) -> Optional[str]:
    if pid not in X2_by_patient:
        return "missing_X"
    if pid not in Xstatic_by_patient:
        return "missing_static"
    if pid not in y_by_patient:
        return "missing_target"

    Xp = X2_by_patient[pid]
    Sp = Xstatic_by_patient[pid]
    yp = y_by_patient[pid]

    if Xp is None:
        return "X_is_none"
    if Sp is None:
        return "static_is_none"
    if yp is None:
        return "target_is_none"

    yp = np.asarray(yp)
    if Xp.ndim != 4 or Xp.shape[1:] != (2, N_TIME, N_DYNAMIC_FEATURES):
        return f"invalid_dynamic_shape_{tuple(Xp.shape)}"
    if Sp.ndim != 2 or Sp.shape[1] != N_STATIC:
        return f"invalid_static_shape_{tuple(Sp.shape)}"
    if Xp.shape[0] == 0 or Sp.shape[0] == 0 or len(yp) == 0:
        return "zero_windows"
    if Xp.shape[0] != Sp.shape[0] or Xp.shape[0] != len(yp):
        return f"length_mismatch_X{Xp.shape[0]}_S{Sp.shape[0]}_y{len(yp)}"
    if not np.isfinite(Xp).all():
        return "nonfinite_dynamic_input"
    if not np.isfinite(Sp).all():
        return "nonfinite_static_input"
    if not np.isfinite(yp.astype(float)).all():
        return "nonfinite_target"
    return None


def build_patient_tables() -> Tuple[pd.DataFrame, pd.DataFrame]:
    valid_rows: List[Dict[str, Any]] = []
    excluded_rows: List[Dict[str, Any]] = []

    candidates = sorted(
        set(X2_by_patient.keys())
        | set(Xstatic_by_patient.keys())
        | set(y_by_patient.keys()),
        key=str,
    )

    for pid in candidates:
        reason = patient_invalid_reason(pid)
        if reason is not None:
            excluded_rows.append({"pid": pid, "exclusion_reason": reason})
            continue
        yp = np.asarray(y_by_patient[pid], dtype=np.float32)
        valid_rows.append(
            {
                "pid": pid,
                "pid_str": str(pid),
                "n_windows": int(len(yp)),
                "target_mean": float(np.mean(yp)),
                "target_median": float(np.median(yp)),
            }
        )

    valid_df = pd.DataFrame(valid_rows)
    if len(valid_df):
        valid_df = valid_df.sort_values("pid_str").reset_index(drop=True)

    excluded_df = pd.DataFrame(excluded_rows)
    if len(excluded_df):
        excluded_df["pid_str"] = excluded_df["pid"].astype(str)
        excluded_df = excluded_df.sort_values("pid_str").reset_index(drop=True)

    return valid_df, excluded_df


patient_df, excluded_patient_df = build_patient_tables()
patient_df.to_csv(os.path.join(EXP_ROOT, "patient_summary_before_cv.csv"), index=False)
excluded_patient_df.to_csv(
    os.path.join(EXP_ROOT, "excluded_patients_with_reason.csv"), index=False
)

if len(patient_df) < OUTER_SPLITS:
    raise ValueError("Not enough valid patients for outer CV.")

# Ensure string conversion identifies each actual PKL key uniquely.
pid_by_string: Dict[str, Any] = {}
for pid in patient_df["pid"].tolist():
    key = str(pid)
    if key in pid_by_string and pid_by_string[key] != pid:
        raise ValueError(f"PID string collision: {pid_by_string[key]!r} vs {pid!r}")
    pid_by_string[key] = pid

print("Valid patients:", len(patient_df))
print("Total windows:", int(patient_df["n_windows"].sum()))


# ============================================================
# 4. Exact main-CNN outer fold reuse
# ============================================================
def load_main_outer_test_sets() -> Dict[int, List[Any]]:
    if not os.path.isfile(MAIN_OUTER_ASSIGNMENT_CSV):
        if REQUIRE_MAIN_CNN_OUTER_FOLDS:
            raise FileNotFoundError(
                "The completed CNN outer-fold assignment was not found:\n"
                f"{MAIN_OUTER_ASSIGNMENT_CSV}\n"
                "Point MAIN_CNN_EXP_ROOT to the completed main run."
            )
        raise RuntimeError("Fallback fold generation is intentionally disabled.")

    assignment = pd.read_csv(MAIN_OUTER_ASSIGNMENT_CSV)
    required = {"pid", "outer_fold", "role"}
    missing = required - set(assignment.columns)
    if missing:
        raise ValueError(
            f"Main fold assignment CSV is missing columns: {sorted(missing)}"
        )

    test_rows = assignment[assignment["role"] == "outer_test"].copy()
    if len(test_rows) == 0:
        raise ValueError("No outer_test rows found in main fold assignment CSV.")

    mapped_rows: List[Dict[str, Any]] = []
    unknown: List[str] = []
    for _, row in test_rows.iterrows():
        pid_text = str(row["pid"])
        # pandas may render integer-like identifiers as '123.0'. Try a safe
        # normalization only when the direct string lookup fails.
        actual = pid_by_string.get(pid_text)
        if actual is None and pid_text.endswith(".0"):
            actual = pid_by_string.get(pid_text[:-2])
        if actual is None:
            unknown.append(pid_text)
            continue
        mapped_rows.append(
            {
                "pid": actual,
                "pid_str": str(actual),
                "outer_fold": int(row["outer_fold"]),
            }
        )

    if unknown:
        raise ValueError(
            "Some patient IDs from the main fold assignment do not match the "
            f"current PKL: {unknown[:20]}"
        )

    mapped = pd.DataFrame(mapped_rows)
    counts = mapped.groupby("pid_str").size()
    if not (counts == 1).all():
        bad = counts[counts != 1]
        raise RuntimeError(
            "Each valid patient must occur once as outer_test in the main "
            f"assignment. Bad counts: {bad.head().to_dict()}"
        )

    expected = set(patient_df["pid_str"])
    observed = set(mapped["pid_str"])
    if expected != observed:
        raise RuntimeError(
            "The current baseline cohort does not match the completed CNN "
            "cohort. Missing from assignment: "
            f"{sorted(expected - observed)[:20]}; unexpected: "
            f"{sorted(observed - expected)[:20]}"
        )

    folds: Dict[int, List[Any]] = {}
    for fold in range(1, OUTER_SPLITS + 1):
        fold_pid_strings = set(
            mapped.loc[mapped["outer_fold"] == fold, "pid_str"].tolist()
        )
        ordered = [
            pid
            for pid in patient_df["pid"].tolist()
            if str(pid) in fold_pid_strings
        ]
        if not ordered:
            raise RuntimeError(f"Outer fold {fold} has no test patients.")
        folds[fold] = ordered

    mapped.sort_values(["outer_fold", "pid_str"]).to_csv(
        os.path.join(EXP_ROOT, "reused_main_outer_test_assignments.csv"),
        index=False,
    )
    return folds


outer_test_sets = load_main_outer_test_sets()
for fold, pids in outer_test_sets.items():
    print(f"Reused outer fold {fold}: {len(pids)} test patients")


# ============================================================
# 5. Same stratification method for inner CV
# ============================================================
def make_strat_bins(
    y_values: Sequence[float],
    n_splits: int,
    max_bins: int = 10,
) -> Tuple[np.ndarray, int]:
    y_values = np.asarray(y_values, dtype=float)
    if not np.isfinite(y_values).all():
        raise ValueError("Non-finite patient targets passed to stratification.")

    for n_bins in range(max_bins, 1, -1):
        try:
            bins = pd.qcut(
                y_values,
                q=n_bins,
                labels=False,
                duplicates="drop",
            )
            bins = np.asarray(bins, dtype=int)
            counts = pd.Series(bins).value_counts()
            if counts.min() >= n_splits:
                return bins, n_bins
        except Exception:
            continue
    return np.zeros(len(y_values), dtype=int), 1


def make_inner_splits(
    outer_fold: int,
    outer_train_df: pd.DataFrame,
) -> List[Tuple[int, List[Any], List[Any]]]:
    inner_bins, used_bins = make_strat_bins(
        outer_train_df["target_mean"].values,
        n_splits=INNER_SPLITS,
        max_bins=8,
    )
    work = outer_train_df.copy()
    work["inner_strat_bin"] = inner_bins

    inner_cv = StratifiedKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=SEED + outer_fold,
    )

    splits: List[Tuple[int, List[Any], List[Any]]] = []
    for inner_fold, (tr_idx, val_idx) in enumerate(
        inner_cv.split(work["pid"].values, work["inner_strat_bin"].values),
        start=1,
    ):
        train_pids = work.iloc[tr_idx]["pid"].tolist()
        val_pids = work.iloc[val_idx]["pid"].tolist()
        splits.append((inner_fold, train_pids, val_pids))

    print(
        f"Outer {outer_fold}: inner stratification uses {used_bins} bins; "
        f"fold sizes={[(len(t), len(v)) for _, t, v in splits]}"
    )
    return splits


# ============================================================
# 6. Patient-balanced normalization and 2D flattening
# ============================================================
@dataclass(frozen=True)
class NormStats:
    dynamic_mean: np.ndarray  # (F,)
    dynamic_std: np.ndarray   # (F,)
    static_mean: np.ndarray   # (S,)
    static_std: np.ndarray    # (S,)


def compute_norm_stats(train_pids: Sequence[Any]) -> NormStats:
    """
    Exactly patient-balanced normalization.

    Each patient contributes one equally weighted dynamic mean vector and one
    equally weighted dynamic squared-deviation vector, irrespective of their
    number of windows. This is algebraically equivalent to giving each window
    weight 1 / n_windows(patient), then averaging across time.
    """
    if not train_pids:
        raise ValueError("Empty train_pids passed to compute_norm_stats.")

    patient_dynamic_means: List[np.ndarray] = []
    patient_static_vectors: List[np.ndarray] = []

    for pid in train_pids:
        Xp = X2_by_patient[pid]
        Sp = Xstatic_by_patient[pid]
        assert Xp is not None and Sp is not None
        vals = Xp[:, 0, :, :].astype(np.float64, copy=False)
        patient_dynamic_means.append(vals.mean(axis=(0, 1)))
        patient_static_vectors.append(Sp[0].astype(np.float64, copy=False))

    dynamic_mean64 = np.mean(np.stack(patient_dynamic_means, axis=0), axis=0)
    static64 = np.stack(patient_static_vectors, axis=0)
    static_mean64 = static64.mean(axis=0)

    patient_dynamic_vars: List[np.ndarray] = []
    for pid in train_pids:
        Xp = X2_by_patient[pid]
        assert Xp is not None
        vals = Xp[:, 0, :, :].astype(np.float64, copy=False)
        patient_dynamic_vars.append(
            ((vals - dynamic_mean64.reshape(1, 1, -1)) ** 2).mean(axis=(0, 1))
        )

    dynamic_var64 = np.mean(np.stack(patient_dynamic_vars, axis=0), axis=0)
    static_var64 = ((static64 - static_mean64) ** 2).mean(axis=0)

    dynamic_std64 = np.sqrt(np.maximum(dynamic_var64, 1e-12))
    static_std64 = np.sqrt(np.maximum(static_var64, 1e-12))

    dynamic_std64 = np.maximum(dynamic_std64, 1e-6)
    static_std64 = np.maximum(static_std64, 1e-6)

    stats = NormStats(
        dynamic_mean=dynamic_mean64.astype(np.float32),
        dynamic_std=dynamic_std64.astype(np.float32),
        static_mean=static_mean64.astype(np.float32),
        static_std=static_std64.astype(np.float32),
    )

    for name, arr, expected in [
        ("dynamic_mean", stats.dynamic_mean, (N_DYNAMIC_FEATURES,)),
        ("dynamic_std", stats.dynamic_std, (N_DYNAMIC_FEATURES,)),
        ("static_mean", stats.static_mean, (N_STATIC,)),
        ("static_std", stats.static_std, (N_STATIC,)),
    ]:
        if arr.shape != expected or not np.isfinite(arr).all():
            raise RuntimeError(f"Invalid {name}: shape={arr.shape}, expected={expected}")
    return stats


def make_unicode_array(values: Sequence[Any]) -> np.ndarray:
    strings = [str(v) for v in values]
    max_len = max(1, max((len(s) for s in strings), default=1))
    return np.asarray(strings, dtype=f"<U{max_len}")


def flatten_patients(
    pids: Sequence[Any],
    stats: NormStats,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    X_parts: List[np.ndarray] = []
    y_parts: List[np.ndarray] = []
    pid_parts: List[np.ndarray] = []
    window_idx_parts: List[np.ndarray] = []

    for pid in pids:
        Xp = X2_by_patient[pid]
        Sp = Xstatic_by_patient[pid]
        yp = np.asarray(y_by_patient[pid], dtype=np.float32)
        assert Xp is not None and Sp is not None

        value_channel = (
            Xp[:, 0, :, :] - stats.dynamic_mean.reshape(1, 1, -1)
        ) / stats.dynamic_std.reshape(1, 1, -1)
        value_flat = value_channel.reshape(len(yp), -1).astype(np.float32)

        parts = [value_flat]
        if INCLUDE_MASK_CHANNEL:
            mask_flat = Xp[:, 1, :, :].reshape(len(yp), -1).astype(np.float32)
            parts.append(mask_flat)

        static_norm = (
            (Sp - stats.static_mean.reshape(1, -1))
            / stats.static_std.reshape(1, -1)
        ).astype(np.float32)
        parts.append(static_norm)

        Zp = np.concatenate(parts, axis=1).astype(np.float32, copy=False)
        if not np.isfinite(Zp).all():
            raise ValueError(f"pid={pid}: non-finite values after flattening.")

        X_parts.append(Zp)
        y_parts.append(yp)
        pid_parts.append(make_unicode_array([pid] * len(yp)))
        window_idx_parts.append(np.arange(len(yp), dtype=np.int32))

    if not X_parts:
        input_dim = (
            N_TIME * N_DYNAMIC_FEATURES
            + (N_TIME * N_DYNAMIC_FEATURES if INCLUDE_MASK_CHANNEL else 0)
            + N_STATIC
        )
        return (
            np.empty((0, input_dim), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype="<U1"),
            np.empty((0,), dtype=np.int32),
        )

    return (
        np.concatenate(X_parts, axis=0).astype(np.float32, copy=False),
        np.concatenate(y_parts, axis=0).astype(np.float32, copy=False),
        np.concatenate(pid_parts, axis=0),
        np.concatenate(window_idx_parts, axis=0),
    )


def patient_balanced_sample_weights(window_pids: np.ndarray) -> np.ndarray:
    pids = np.asarray(window_pids).astype(str)
    if len(pids) == 0:
        return np.empty((0,), dtype=np.float64)
    counts = Counter(pids.tolist())
    weights = np.asarray([1.0 / counts[pid] for pid in pids], dtype=np.float64)
    totals = {
        pid: float(weights[pids == pid].sum())
        for pid in counts
    }
    if not np.allclose(list(totals.values()), 1.0, rtol=0.0, atol=1e-10):
        raise RuntimeError("Patient-balanced sample weights failed validation.")
    return weights


# Save a human-readable flattened-column map.
flattened_columns: List[Dict[str, Any]] = []
col_index = 0
for t in range(N_TIME):
    for feature in dynamic_feature_cols:
        flattened_columns.append(
            {
                "column_index": col_index,
                "source": "value_channel",
                "time_index": t,
                "feature": feature,
            }
        )
        col_index += 1
if INCLUDE_MASK_CHANNEL:
    for t in range(N_TIME):
        for feature in dynamic_feature_cols:
            flattened_columns.append(
                {
                    "column_index": col_index,
                    "source": "mask_channel",
                    "time_index": t,
                    "feature": feature,
                }
            )
            col_index += 1
for feature in static_feature_cols:
    flattened_columns.append(
        {
            "column_index": col_index,
            "source": "static",
            "time_index": np.nan,
            "feature": feature,
        }
    )
    col_index += 1
pd.DataFrame(flattened_columns).to_csv(
    os.path.join(EXP_ROOT, "flattened_2d_column_map.csv"), index=False
)


# ============================================================
# 7. Persistent prepared split cache
# ============================================================
@dataclass(frozen=True)
class PreparedSplit:
    cache_dir: str
    split_hash: str

    def path(self, name: str) -> str:
        return os.path.join(self.cache_dir, f"{name}.npy")

    def load(self, name: str, mmap: bool = True) -> np.ndarray:
        return np.load(self.path(name), mmap_mode="r" if mmap else None)

    @property
    def meta(self) -> Dict[str, Any]:
        return read_json(os.path.join(self.cache_dir, "meta.json"))


def ensure_prepared_split(
    cache_dir: str,
    split_name: str,
    train_pids: Sequence[Any],
    eval_pids: Sequence[Any],
    parent_identity: Mapping[str, Any],
) -> PreparedSplit:
    os.makedirs(cache_dir, exist_ok=True)

    split_identity = {
        **dict(parent_identity),
        "split_name": split_name,
        "train_pids": [str(x) for x in train_pids],
        "eval_pids": [str(x) for x in eval_pids],
        "normalization": NORMALIZATION_MODE,
        "include_mask_channel": INCLUDE_MASK_CHANNEL,
        "dynamic_features": dynamic_feature_cols,
        "static_features": static_feature_cols,
    }
    split_hash = stable_hash(split_identity)
    meta_path = os.path.join(cache_dir, "meta.json")

    required_arrays = [
        "X_train",
        "y_train",
        "w_train",
        "train_pids",
        "train_window_idx",
        "X_eval",
        "y_eval",
        "eval_pids",
        "eval_window_idx",
        "dynamic_mean",
        "dynamic_std",
        "static_mean",
        "static_std",
    ]
    required_paths = [os.path.join(cache_dir, f"{name}.npy") for name in required_arrays]

    if os.path.isfile(meta_path):
        meta = read_json(meta_path)
        if meta.get("split_hash") != split_hash:
            raise RuntimeError(
                f"Prepared split cache mismatch at {cache_dir}. "
                "Use a new EXP_ROOT or delete only this incompatible cache."
            )
        missing = [p for p in required_paths if not os.path.isfile(p)]
        if not missing:
            print(f"Using prepared cache: {split_name}")
            return PreparedSplit(cache_dir=cache_dir, split_hash=split_hash)

    print(f"Preparing split cache: {split_name}")
    stats = compute_norm_stats(train_pids)
    X_train, y_train, train_window_pids, train_window_idx = flatten_patients(
        train_pids, stats
    )
    X_eval, y_eval, eval_window_pids, eval_window_idx = flatten_patients(
        eval_pids, stats
    )
    w_train = patient_balanced_sample_weights(train_window_pids)

    if X_train.shape[0] == 0 or X_eval.shape[0] == 0:
        raise ValueError(f"{split_name}: empty train or evaluation matrix.")
    if X_train.shape[1] != X_eval.shape[1]:
        raise RuntimeError(f"{split_name}: train/eval input dimensions differ.")

    arrays = {
        "X_train": X_train,
        "y_train": y_train,
        "w_train": w_train,
        "train_pids": train_window_pids,
        "train_window_idx": train_window_idx,
        "X_eval": X_eval,
        "y_eval": y_eval,
        "eval_pids": eval_window_pids,
        "eval_window_idx": eval_window_idx,
        "dynamic_mean": stats.dynamic_mean,
        "dynamic_std": stats.dynamic_std,
        "static_mean": stats.static_mean,
        "static_std": stats.static_std,
    }
    for name, arr in arrays.items():
        atomic_npy_save(np.asarray(arr), os.path.join(cache_dir, f"{name}.npy"))

    meta = {
        "split_hash": split_hash,
        "split_identity": split_identity,
        "n_train_patients": len(train_pids),
        "n_eval_patients": len(eval_pids),
        "n_train_windows": int(len(y_train)),
        "n_eval_windows": int(len(y_eval)),
        "input_dim": int(X_train.shape[1]),
        "train_matrix_size_gib": float(X_train.nbytes / (1024 ** 3)),
        "eval_matrix_size_gib": float(X_eval.nbytes / (1024 ** 3)),
    }
    atomic_json_dump(meta, meta_path)

    del X_train, X_eval, y_train, y_eval, w_train
    cleanup()
    return PreparedSplit(cache_dir=cache_dir, split_hash=split_hash)


# ============================================================
# 8. Evaluation utilities
# ============================================================
def safe_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(y_true) < 2 or np.allclose(y_true, y_true[0]):
        return float("nan")
    return float(r2_score(y_true, y_pred))


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_pred - y_true
    return {
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "r2": safe_r2(y_true, y_pred),
        "n": int(len(y_true)),
    }


def patient_prediction_table(
    y_true_window: np.ndarray,
    y_pred_window: np.ndarray,
    window_pids: np.ndarray,
    outer_fold: Optional[int] = None,
    model_name: Optional[str] = None,
    seed: Optional[int] = None,
) -> pd.DataFrame:
    frame = pd.DataFrame(
        {
            "pid": np.asarray(window_pids).astype(str),
            "true_window": np.asarray(y_true_window, dtype=float),
            "pred_window": np.asarray(y_pred_window, dtype=float),
        }
    )
    pat = (
        frame.groupby("pid", as_index=False)
        .agg(
            true_patient=("true_window", "mean"),
            pred_patient=("pred_window", "mean"),
            pred_window_std=("pred_window", "std"),
            true_window_std=("true_window", "std"),
            n_windows=("true_window", "size"),
        )
    )
    pat["pred_window_std"] = pat["pred_window_std"].fillna(0.0)
    pat["true_window_std"] = pat["true_window_std"].fillna(0.0)
    pat["error_patient"] = pat["pred_patient"] - pat["true_patient"]
    pat["abs_error"] = np.abs(pat["error_patient"])
    pat["sq_error"] = pat["error_patient"] ** 2
    if outer_fold is not None:
        pat["outer_fold"] = int(outer_fold)
    if model_name is not None:
        pat["model"] = model_name
    if seed is not None:
        pat["seed"] = int(seed)
    return pat


def metrics_from_prepared_eval(
    prepared: PreparedSplit,
    predictions: np.ndarray,
) -> Tuple[Dict[str, float], Dict[str, float], pd.DataFrame]:
    y_eval = np.asarray(prepared.load("y_eval"), dtype=np.float32)
    eval_pids = np.asarray(prepared.load("eval_pids"), dtype=str)
    predictions = np.asarray(predictions, dtype=np.float32)
    if len(predictions) != len(y_eval):
        raise ValueError("Prediction length does not match evaluation targets.")

    window_metrics = regression_metrics(y_eval, predictions)
    patient_df_local = patient_prediction_table(y_eval, predictions, eval_pids)
    patient_metrics = regression_metrics(
        patient_df_local["true_patient"].values,
        patient_df_local["pred_patient"].values,
    )
    return window_metrics, patient_metrics, patient_df_local


# ============================================================
# 9. Fixed model families and Optuna hyperparameter spaces
# ============================================================
def sample_model_params(model_name: str, trial: optuna.Trial) -> Dict[str, Any]:
    if model_name == "ridge":
        # Ridge is used instead of unregularized OLS because the flattened 2D
        # representation is high-dimensional and strongly collinear.
        return {
            "alpha": trial.suggest_float("alpha", 1e-5, 1e5, log=True),
            "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        }

    if model_name == "xgboost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1600, step=100),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.005, 0.20, log=True
            ),
            "min_child_weight": trial.suggest_float(
                "min_child_weight", 0.5, 40.0, log=True
            ),
            "subsample": trial.suggest_float("subsample", 0.60, 1.00),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.35, 1.00
            ),
            "gamma": trial.suggest_float("gamma", 0.0, 3.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 20.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
            "max_bin": trial.suggest_categorical("max_bin", [128, 256, 512]),
        }

    raise ValueError(f"Unknown model: {model_name}")


def xgb_version_major() -> int:
    try:
        return int(str(xgb.__version__).split(".")[0])
    except Exception:
        return 2


def build_model(
    model_name: str,
    params: Mapping[str, Any],
    seed: int,
):
    if model_name == "ridge":
        return Ridge(
            alpha=float(params["alpha"]),
            fit_intercept=True,
            solver="lsqr",
            tol=float(params.get("tol", 1e-4)),
            max_iter=20000,
        )

    if model_name == "xgboost":
        kwargs: Dict[str, Any] = {
            **dict(params),
            "objective": "reg:squarederror",
            "eval_metric": "mae",
            "random_state": int(seed),
            "n_jobs": int(XGB_N_JOBS),
            "tree_method": "hist",
            "verbosity": 0,
        }
        if XGB_DEVICE == "cuda":
            if xgb_version_major() >= 2:
                kwargs["device"] = "cuda"
            else:
                kwargs["tree_method"] = "gpu_hist"
                kwargs["predictor"] = "gpu_predictor"
        elif XGB_DEVICE != "cpu":
            raise ValueError("XGB_DEVICE must be 'cuda' or 'cpu'.")
        return XGBRegressor(**kwargs)

    raise ValueError(f"Unknown model: {model_name}")


def fit_model(
    model_name: str,
    params: Mapping[str, Any],
    prepared: PreparedSplit,
    seed: int,
):
    X_train = prepared.load("X_train")
    y_train = np.asarray(prepared.load("y_train"), dtype=np.float32)
    w_train = np.asarray(prepared.load("w_train"), dtype=np.float64)

    model = build_model(model_name, params, seed)
    model.fit(X_train, y_train, sample_weight=w_train)
    return model


# ============================================================
# 10. Experiment identity
# ============================================================
if "__file__" in globals() and os.path.isfile(__file__):
    with open(__file__, "rb") as f:
        SCRIPT_SHA256 = hashlib.sha256(f.read()).hexdigest()
else:
    SCRIPT_SHA256 = "unavailable"

stat = os.stat(DATA_PKL)
dataset_fingerprint = {
    "absolute_path": os.path.abspath(DATA_PKL),
    "size_bytes": int(stat.st_size),
    "mtime_ns": int(stat.st_mtime_ns),
}
main_assignment_stat = os.stat(MAIN_OUTER_ASSIGNMENT_CSV)
main_assignment_fingerprint = {
    "absolute_path": os.path.abspath(MAIN_OUTER_ASSIGNMENT_CSV),
    "size_bytes": int(main_assignment_stat.st_size),
    "mtime_ns": int(main_assignment_stat.st_mtime_ns),
}

run_config = {
    "code_version": CODE_VERSION,
    "script_sha256": SCRIPT_SHA256,
    "dataset_label": DATASET_LABEL,
    "dataset_fingerprint": dataset_fingerprint,
    "main_assignment_fingerprint": main_assignment_fingerprint,
    "outer_splits": OUTER_SPLITS,
    "inner_splits": INNER_SPLITS,
    "models": list(MODELS_TO_RUN),
    "n_trials_per_outer": N_TRIALS_PER_OUTER,
    "final_refit_seeds": FINAL_REFIT_SEEDS,
    "normalization": NORMALIZATION_MODE,
    "patient_balanced_training": PATIENT_BALANCED_TRAINING,
    "include_mask_channel": INCLUDE_MASK_CHANNEL,
    "flattening": "value_channel_then_mask_channel_then_static_C_order",
    "static_features": static_feature_cols,
    "dynamic_features": dynamic_feature_cols,
    "forbidden_features": FORBIDDEN_FEATURES,
    "xgboost_version": xgb.__version__,
    "xgb_device": XGB_DEVICE,
}
EXPERIMENT_HASH = stable_hash(run_config)
run_config["experiment_hash"] = EXPERIMENT_HASH

signature_path = os.path.join(EXP_ROOT, "experiment_signature.json")
if os.path.isfile(signature_path):
    existing = read_json(signature_path)
    if existing.get("experiment_hash") != EXPERIMENT_HASH:
        raise RuntimeError(
            "EXP_ROOT contains an incompatible experiment. Use a new EXP_ROOT "
            "or restore the matching script/configuration."
        )
else:
    atomic_json_dump(run_config, signature_path)
atomic_json_dump(run_config, os.path.join(EXP_ROOT, "run_config.json"))
print("EXPERIMENT_HASH:", EXPERIMENT_HASH)


# ============================================================
# 11. Optuna study helpers
# ============================================================
def trial_state_counts(study: optuna.Study) -> Dict[str, int]:
    counts = Counter(trial.state.name for trial in study.trials)
    return {state: int(counts.get(state, 0)) for state in [
        "COMPLETE", "PRUNED", "FAIL", "RUNNING", "WAITING"
    ]}


def top_up_study(
    study: optuna.Study,
    objective,
    target_count: int,
) -> None:
    while True:
        counts = trial_state_counts(study)
        counted = counts["COMPLETE"] + counts["PRUNED"]
        remaining = int(target_count - counted)
        print(
            f"Study {study.study_name}: COMPLETE={counts['COMPLETE']}, "
            f"PRUNED={counts['PRUNED']}, FAIL={counts['FAIL']}, "
            f"COUNTED={counted}/{target_count}, REMAINING={remaining}"
        )
        if remaining <= 0:
            break

        before = counted
        catch_types: Tuple[type, ...] = (
            RuntimeError,
            MemoryError,
            xgb.core.XGBoostError,
        )
        study.optimize(
            objective,
            n_trials=remaining,
            gc_after_trial=True,
            catch=catch_types,
        )
        counts_after = trial_state_counts(study)
        after = counts_after["COMPLETE"] + counts_after["PRUNED"]
        if after <= before:
            raise RuntimeError(
                f"Study {study.study_name} made no progress. Inspect failed trials."
            )


def save_study_outputs(study: optuna.Study, output_dir: str) -> None:
    trials_df = study.trials_dataframe()
    trials_df.to_csv(os.path.join(output_dir, "optuna_trials.csv"), index=False)
    best_payload = {
        "best_trial_number": int(study.best_trial.number),
        "best_value_inner_patient_mae": float(study.best_value),
        "best_params": study.best_params,
        "best_user_attrs": study.best_trial.user_attrs,
        "state_counts": trial_state_counts(study),
    }
    atomic_json_dump(best_payload, os.path.join(output_dir, "optuna_best.json"))


# ============================================================
# 12. Final prediction aggregation
# ============================================================
def make_window_prediction_df(
    prepared: PreparedSplit,
    predictions: np.ndarray,
    outer_fold: int,
    model_name: str,
    seed: int,
) -> pd.DataFrame:
    y_eval = np.asarray(prepared.load("y_eval"), dtype=np.float32)
    eval_pids = np.asarray(prepared.load("eval_pids"), dtype=str)
    eval_window_idx = np.asarray(prepared.load("eval_window_idx"), dtype=np.int32)
    pred = np.asarray(predictions, dtype=np.float32)
    if len(pred) != len(y_eval):
        raise ValueError("Prediction length mismatch.")
    df = pd.DataFrame(
        {
            "pid": eval_pids,
            "window_idx": eval_window_idx,
            "window_id": [f"{pid}_{idx}" for pid, idx in zip(eval_pids, eval_window_idx)],
            "true_window": y_eval,
            "pred_window": pred,
            "error_window": pred - y_eval,
            "outer_fold": int(outer_fold),
            "model": model_name,
            "seed": int(seed),
        }
    )
    return df


def ensemble_seed_window_predictions(seed_predictions: pd.DataFrame) -> pd.DataFrame:
    if len(seed_predictions) == 0:
        return pd.DataFrame()
    keys = ["pid", "window_idx", "window_id", "outer_fold", "model"]
    true_counts = seed_predictions.groupby(keys)["true_window"].nunique()
    if not (true_counts == 1).all():
        raise RuntimeError("Inconsistent true_window values across final seeds.")
    out = (
        seed_predictions.groupby(keys, as_index=False)
        .agg(
            true_window=("true_window", "first"),
            pred_window_mean=("pred_window", "mean"),
            pred_window_std_across_seeds=("pred_window", "std"),
            n_seeds=("pred_window", "size"),
        )
    )
    out["pred_window_std_across_seeds"] = out[
        "pred_window_std_across_seeds"
    ].fillna(0.0)
    out["error_window"] = out["pred_window_mean"] - out["true_window"]
    out["abs_error_window"] = np.abs(out["error_window"])
    return out


def patient_from_ensemble_windows(window_df: pd.DataFrame) -> pd.DataFrame:
    pat = (
        window_df.groupby(["pid", "outer_fold", "model"], as_index=False)
        .agg(
            true_patient=("true_window", "mean"),
            pred_patient=("pred_window_mean", "mean"),
            pred_patient_std_across_windows=("pred_window_mean", "std"),
            n_windows=("window_idx", "size"),
            n_seeds=("n_seeds", "first"),
        )
    )
    pat["pred_patient_std_across_windows"] = pat[
        "pred_patient_std_across_windows"
    ].fillna(0.0)
    pat["error_patient"] = pat["pred_patient"] - pat["true_patient"]
    pat["abs_error"] = np.abs(pat["error_patient"])
    pat["sq_error"] = pat["error_patient"] ** 2
    return pat


# ============================================================
# 13. Run nested CV
# ============================================================
all_model_outer_windows: Dict[str, List[pd.DataFrame]] = {
    model: [] for model in MODELS_TO_RUN
}
all_model_outer_patients: Dict[str, List[pd.DataFrame]] = {
    model: [] for model in MODELS_TO_RUN
}
outer_summary_rows: List[Dict[str, Any]] = []

all_ordered_pids = patient_df["pid"].tolist()

for outer_fold in range(1, OUTER_SPLITS + 1):
    outer_dir = os.path.join(EXP_ROOT, f"outer_fold_{outer_fold:02d}")
    os.makedirs(outer_dir, exist_ok=True)

    outer_test_pids = outer_test_sets[outer_fold]
    outer_test_set_str = {str(pid) for pid in outer_test_pids}
    outer_train_pids = [
        pid for pid in all_ordered_pids if str(pid) not in outer_test_set_str
    ]

    outer_train_df = patient_df[
        patient_df["pid_str"].isin({str(pid) for pid in outer_train_pids})
    ].copy().reset_index(drop=True)
    outer_test_df = patient_df[
        patient_df["pid_str"].isin(outer_test_set_str)
    ].copy().reset_index(drop=True)

    if set(outer_train_df["pid_str"]) & set(outer_test_df["pid_str"]):
        raise RuntimeError(f"Outer fold {outer_fold}: train/test patient overlap.")

    outer_identity = {
        "experiment_hash": EXPERIMENT_HASH,
        "outer_fold": outer_fold,
        "outer_train_pids": [str(x) for x in outer_train_pids],
        "outer_test_pids": [str(x) for x in outer_test_pids],
    }
    outer_hash = stable_hash(outer_identity)

    print("\n" + "=" * 110)
    print(
        f"OUTER FOLD {outer_fold}/{OUTER_SPLITS}: "
        f"train patients={len(outer_train_pids)}, "
        f"test patients={len(outer_test_pids)}"
    )
    print("=" * 110)

    inner_splits = make_inner_splits(outer_fold, outer_train_df)
    inner_assignment_rows: List[Dict[str, Any]] = []
    for inner_fold, train_pids, val_pids in inner_splits:
        for pid in train_pids:
            inner_assignment_rows.append(
                {"inner_fold": inner_fold, "pid": str(pid), "role": "inner_train"}
            )
        for pid in val_pids:
            inner_assignment_rows.append(
                {"inner_fold": inner_fold, "pid": str(pid), "role": "inner_val"}
            )
    pd.DataFrame(inner_assignment_rows).to_csv(
        os.path.join(outer_dir, "inner_fold_assignment_long.csv"), index=False
    )

    prepared_root = os.path.join(outer_dir, "prepared_2d_flattened")
    prepared_inner: List[Tuple[int, PreparedSplit]] = []
    for inner_fold, inner_train_pids, inner_val_pids in inner_splits:
        prepared = ensure_prepared_split(
            cache_dir=os.path.join(prepared_root, f"inner_fold_{inner_fold:02d}"),
            split_name=f"outer{outer_fold}_inner{inner_fold}",
            train_pids=inner_train_pids,
            eval_pids=inner_val_pids,
            parent_identity={"outer_hash": outer_hash, "inner_fold": inner_fold},
        )
        prepared_inner.append((inner_fold, prepared))

    prepared_final = ensure_prepared_split(
        cache_dir=os.path.join(prepared_root, "full_outer_train_to_test"),
        split_name=f"outer{outer_fold}_full_train_to_test",
        train_pids=outer_train_pids,
        eval_pids=outer_test_pids,
        parent_identity={"outer_hash": outer_hash, "stage": "final_refit"},
    )

    for model_name in MODELS_TO_RUN:
        model_dir = os.path.join(outer_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)
        complete_marker = os.path.join(model_dir, "MODEL_OUTER_COMPLETE.json")
        ensemble_window_csv = os.path.join(
            model_dir, "outer_test_window_predictions_ensemble.csv"
        )
        ensemble_patient_csv = os.path.join(
            model_dir, "outer_test_patient_predictions_ensemble.csv"
        )

        if os.path.isfile(complete_marker):
            marker = read_json(complete_marker)
            if (
                marker.get("experiment_hash") != EXPERIMENT_HASH
                or marker.get("outer_hash") != outer_hash
                or marker.get("model") != model_name
            ):
                raise RuntimeError(f"Incompatible completion marker: {complete_marker}")
            if not os.path.isfile(ensemble_window_csv) or not os.path.isfile(
                ensemble_patient_csv
            ):
                raise RuntimeError(
                    f"Completion marker exists but prediction CSVs are missing: {model_dir}"
                )
            loaded_w = pd.read_csv(ensemble_window_csv, dtype={"pid": str})
            loaded_p = pd.read_csv(ensemble_patient_csv, dtype={"pid": str})
            all_model_outer_windows[model_name].append(loaded_w)
            all_model_outer_patients[model_name].append(loaded_p)
            outer_summary_rows.append(marker["outer_summary"])
            print(f"Outer {outer_fold} {model_name}: already complete; loaded outputs.")
            continue

        def objective(trial: optuna.Trial) -> float:
            params = sample_model_params(model_name, trial)
            patient_maes: List[float] = []
            patient_rmses: List[float] = []
            patient_r2s: List[float] = []
            window_maes: List[float] = []

            for inner_fold, prepared in prepared_inner:
                run_seed = SEED + outer_fold * 100 + inner_fold
                model = fit_model(model_name, params, prepared, run_seed)
                X_eval = prepared.load("X_eval")
                pred = np.asarray(model.predict(X_eval), dtype=np.float32)
                window_metrics, patient_metrics, _ = metrics_from_prepared_eval(
                    prepared, pred
                )

                patient_maes.append(patient_metrics["mae"])
                patient_rmses.append(patient_metrics["rmse"])
                patient_r2s.append(patient_metrics["r2"])
                window_maes.append(window_metrics["mae"])

                trial.report(float(np.mean(patient_maes)), step=inner_fold)
                del model, pred
                cleanup()

                if trial.should_prune():
                    raise optuna.TrialPruned()

            trial.set_user_attr(
                "inner_val_patient_MAE_mean", float(np.mean(patient_maes))
            )
            trial.set_user_attr(
                "inner_val_patient_MAE_std", float(np.std(patient_maes, ddof=0))
            )
            trial.set_user_attr(
                "inner_val_patient_RMSE_mean", float(np.mean(patient_rmses))
            )
            trial.set_user_attr(
                "inner_val_patient_R2_mean", float(np.nanmean(patient_r2s))
            )
            trial.set_user_attr(
                "inner_val_window_MAE_mean", float(np.mean(window_maes))
            )
            trial.set_user_attr(
                "inner_fold_patient_MAEs", [float(v) for v in patient_maes]
            )
            return float(np.mean(patient_maes))

        sampler = optuna.samplers.TPESampler(seed=SEED + outer_fold)
        if model_name == "xgboost":
            pruner: optuna.pruners.BasePruner = optuna.pruners.MedianPruner(
                n_startup_trials=10,
                n_warmup_steps=2,
            )
        else:
            pruner = optuna.pruners.NopPruner()

        study = optuna.create_study(
            study_name=f"2d_flattened_{model_name}_outer_{outer_fold:02d}",
            storage="sqlite:///" + os.path.join(model_dir, "optuna_study.db"),
            load_if_exists=True,
            direction="minimize",
            sampler=sampler,
            pruner=pruner,
        )

        existing_hash = study.user_attrs.get("experiment_hash")
        existing_outer_hash = study.user_attrs.get("outer_hash")
        existing_model = study.user_attrs.get("model")
        if existing_hash is None:
            if len(study.trials) > 0:
                raise RuntimeError(
                    f"Existing unsigned Optuna study in {model_dir}; use a new output path."
                )
            study.set_user_attr("experiment_hash", EXPERIMENT_HASH)
            study.set_user_attr("outer_hash", outer_hash)
            study.set_user_attr("model", model_name)
        elif (
            existing_hash != EXPERIMENT_HASH
            or existing_outer_hash != outer_hash
            or existing_model != model_name
        ):
            raise RuntimeError(f"Optuna study identity mismatch in {model_dir}")

        top_up_study(
            study,
            objective,
            target_count=N_TRIALS_PER_OUTER[model_name],
        )

        if not any(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials):
            raise RuntimeError(f"No complete Optuna trial for {model_name}, outer {outer_fold}")

        save_study_outputs(study, model_dir)
        best_params = dict(study.best_params)
        print(
            f"Outer {outer_fold} {model_name}: best trial={study.best_trial.number}, "
            f"inner patient MAE={study.best_value:.4f}, params={best_params}"
        )

        seed_window_frames: List[pd.DataFrame] = []
        seed_patient_frames: List[pd.DataFrame] = []
        seed_metric_rows: List[Dict[str, Any]] = []

        for seed in FINAL_REFIT_SEEDS[model_name]:
            seed_dir = os.path.join(model_dir, f"seed_{seed}")
            os.makedirs(seed_dir, exist_ok=True)
            seed_marker = os.path.join(seed_dir, "SEED_COMPLETE.json")
            seed_window_csv = os.path.join(seed_dir, "outer_test_window_predictions.csv")
            seed_patient_csv = os.path.join(seed_dir, "outer_test_patient_predictions.csv")
            model_path = os.path.join(seed_dir, "fitted_model.joblib")

            if os.path.isfile(seed_marker):
                marker = read_json(seed_marker)
                expected_identity = {
                    "experiment_hash": EXPERIMENT_HASH,
                    "outer_hash": outer_hash,
                    "model": model_name,
                    "seed": int(seed),
                    "best_params_hash": stable_hash(best_params),
                }
                for key, value in expected_identity.items():
                    if marker.get(key) != value:
                        raise RuntimeError(f"Seed marker mismatch: {seed_marker}")
                if not os.path.isfile(seed_window_csv) or not os.path.isfile(
                    seed_patient_csv
                ):
                    raise RuntimeError(f"Seed marker exists but CSVs are missing: {seed_dir}")
                window_df_seed = pd.read_csv(seed_window_csv, dtype={"pid": str})
                patient_df_seed = pd.read_csv(seed_patient_csv, dtype={"pid": str})
                seed_metrics = marker["metrics"]
                print(f"Outer {outer_fold} {model_name} seed {seed}: loaded.")
            else:
                model = fit_model(model_name, best_params, prepared_final, seed)
                X_test = prepared_final.load("X_eval")
                pred = np.asarray(model.predict(X_test), dtype=np.float32)

                window_df_seed = make_window_prediction_df(
                    prepared_final,
                    pred,
                    outer_fold=outer_fold,
                    model_name=model_name,
                    seed=seed,
                )
                patient_df_seed = patient_prediction_table(
                    window_df_seed["true_window"].values,
                    window_df_seed["pred_window"].values,
                    window_df_seed["pid"].values,
                    outer_fold=outer_fold,
                    model_name=model_name,
                    seed=seed,
                )

                window_metrics = regression_metrics(
                    window_df_seed["true_window"].values,
                    window_df_seed["pred_window"].values,
                )
                patient_metrics = regression_metrics(
                    patient_df_seed["true_patient"].values,
                    patient_df_seed["pred_patient"].values,
                )
                seed_metrics = {
                    "window": window_metrics,
                    "patient": patient_metrics,
                }

                window_df_seed.to_csv(seed_window_csv, index=False)
                patient_df_seed.to_csv(seed_patient_csv, index=False)
                atomic_joblib_dump(model, model_path)
                atomic_json_dump(
                    {
                        "experiment_hash": EXPERIMENT_HASH,
                        "outer_hash": outer_hash,
                        "model": model_name,
                        "seed": int(seed),
                        "best_params_hash": stable_hash(best_params),
                        "best_params": best_params,
                        "metrics": seed_metrics,
                        "model_path": model_path,
                    },
                    seed_marker,
                )
                del model, pred
                cleanup()

            seed_window_frames.append(window_df_seed)
            seed_patient_frames.append(patient_df_seed)
            seed_metric_rows.append(
                {
                    "outer_fold": outer_fold,
                    "model": model_name,
                    "seed": int(seed),
                    "patient_mae": seed_metrics["patient"]["mae"],
                    "patient_rmse": seed_metrics["patient"]["rmse"],
                    "patient_r2": seed_metrics["patient"]["r2"],
                    "window_mae": seed_metrics["window"]["mae"],
                    "window_rmse": seed_metrics["window"]["rmse"],
                    "window_r2": seed_metrics["window"]["r2"],
                }
            )

        pd.DataFrame(seed_metric_rows).to_csv(
            os.path.join(model_dir, "final_seed_metrics.csv"), index=False
        )
        all_seed_windows = pd.concat(seed_window_frames, ignore_index=True)
        all_seed_windows.to_csv(
            os.path.join(model_dir, "outer_test_window_predictions_all_seeds.csv"),
            index=False,
        )
        all_seed_patients = pd.concat(seed_patient_frames, ignore_index=True)
        all_seed_patients.to_csv(
            os.path.join(model_dir, "outer_test_patient_predictions_all_seeds.csv"),
            index=False,
        )

        ensemble_window = ensemble_seed_window_predictions(all_seed_windows)
        ensemble_patient = patient_from_ensemble_windows(ensemble_window)
        ensemble_window.to_csv(ensemble_window_csv, index=False)
        ensemble_patient.to_csv(ensemble_patient_csv, index=False)

        ensemble_window_metrics = regression_metrics(
            ensemble_window["true_window"].values,
            ensemble_window["pred_window_mean"].values,
        )
        ensemble_patient_metrics = regression_metrics(
            ensemble_patient["true_patient"].values,
            ensemble_patient["pred_patient"].values,
        )

        outer_summary = {
            "outer_fold": outer_fold,
            "model": model_name,
            "n_outer_train_patients": len(outer_train_pids),
            "n_outer_test_patients": len(outer_test_pids),
            "n_outer_train_windows": int(prepared_final.meta["n_train_windows"]),
            "n_outer_test_windows": int(prepared_final.meta["n_eval_windows"]),
            "input_dim": int(prepared_final.meta["input_dim"]),
            "n_optuna_trials_target": int(N_TRIALS_PER_OUTER[model_name]),
            "best_trial": int(study.best_trial.number),
            "best_inner_patient_mae": float(study.best_value),
            "best_params_json": json.dumps(json_safe(best_params), sort_keys=True),
            "final_refit_seeds": ";".join(map(str, FINAL_REFIT_SEEDS[model_name])),
            "n_final_refit_seeds": len(FINAL_REFIT_SEEDS[model_name]),
            "outer_test_patient_mae": ensemble_patient_metrics["mae"],
            "outer_test_patient_rmse": ensemble_patient_metrics["rmse"],
            "outer_test_patient_r2": ensemble_patient_metrics["r2"],
            "outer_test_window_mae": ensemble_window_metrics["mae"],
            "outer_test_window_rmse": ensemble_window_metrics["rmse"],
            "outer_test_window_r2": ensemble_window_metrics["r2"],
        }
        outer_summary_rows.append(outer_summary)
        all_model_outer_windows[model_name].append(ensemble_window)
        all_model_outer_patients[model_name].append(ensemble_patient)

        atomic_json_dump(
            {
                "experiment_hash": EXPERIMENT_HASH,
                "outer_hash": outer_hash,
                "model": model_name,
                "outer_summary": outer_summary,
            },
            complete_marker,
        )
        pd.DataFrame(outer_summary_rows).to_csv(
            os.path.join(EXP_ROOT, "outer_fold_summary_running.csv"), index=False
        )


# ============================================================
# 14. Pooled nested OOF results
# ============================================================
outer_summary_df = pd.DataFrame(outer_summary_rows).sort_values(
    ["model", "outer_fold"]
)
outer_summary_df.to_csv(os.path.join(EXP_ROOT, "outer_fold_summary.csv"), index=False)

pooled_summary_rows: List[Dict[str, Any]] = []
pooled_patient_frames: List[pd.DataFrame] = []

for model_name in MODELS_TO_RUN:
    pooled_window = pd.concat(all_model_outer_windows[model_name], ignore_index=True)
    pooled_patient = pd.concat(all_model_outer_patients[model_name], ignore_index=True)

    # Strict OOF integrity: each valid patient appears once and in one outer fold.
    expected_pids = set(patient_df["pid_str"])
    observed_pids = set(pooled_patient["pid"].astype(str))
    if expected_pids != observed_pids:
        raise RuntimeError(
            f"{model_name}: pooled OOF patient mismatch. Missing="
            f"{sorted(expected_pids - observed_pids)[:20]}, unexpected="
            f"{sorted(observed_pids - expected_pids)[:20]}"
        )
    fold_counts = pooled_patient.groupby("pid")["outer_fold"].nunique()
    if not (fold_counts == 1).all():
        raise RuntimeError(f"{model_name}: a patient appears in multiple test folds.")
    duplicate_windows = pooled_window.duplicated(
        subset=["pid", "window_idx"], keep=False
    )
    if duplicate_windows.any():
        raise RuntimeError(f"{model_name}: duplicated OOF patient-window rows.")

    pooled_window = pooled_window.sort_values(
        ["outer_fold", "pid", "window_idx"]
    ).reset_index(drop=True)
    pooled_patient = pooled_patient.sort_values(
        "abs_error", ascending=False
    ).reset_index(drop=True)

    pooled_window.to_csv(
        os.path.join(EXP_ROOT, f"FINAL_nested_oof_window_predictions_{model_name}.csv"),
        index=False,
    )
    pooled_patient.to_csv(
        os.path.join(EXP_ROOT, f"FINAL_nested_oof_patient_predictions_{model_name}.csv"),
        index=False,
    )

    window_metrics = regression_metrics(
        pooled_window["true_window"].values,
        pooled_window["pred_window_mean"].values,
    )
    patient_metrics = regression_metrics(
        pooled_patient["true_patient"].values,
        pooled_patient["pred_patient"].values,
    )

    summary_row = {
        "model": model_name,
        "input": "120min_2D_flattened_value_plus_mask_plus_static",
        "n_patients": patient_metrics["n"],
        "n_windows": window_metrics["n"],
        "patient_mae": patient_metrics["mae"],
        "patient_rmse": patient_metrics["rmse"],
        "patient_r2": patient_metrics["r2"],
        "window_mae": window_metrics["mae"],
        "window_rmse": window_metrics["rmse"],
        "window_r2": window_metrics["r2"],
        "outer_fold_patient_mae_mean": float(
            outer_summary_df.loc[
                outer_summary_df["model"] == model_name,
                "outer_test_patient_mae",
            ].mean()
        ),
        "outer_fold_patient_mae_std": float(
            outer_summary_df.loc[
                outer_summary_df["model"] == model_name,
                "outer_test_patient_mae",
            ].std(ddof=0)
        ),
    }
    pooled_summary_rows.append(summary_row)
    pooled_patient_frames.append(pooled_patient.copy())

    # Patient-level predicted-vs-true plot.
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.scatter(
        pooled_patient["true_patient"],
        pooled_patient["pred_patient"],
        alpha=0.7,
    )
    low = float(
        min(pooled_patient["true_patient"].min(), pooled_patient["pred_patient"].min())
    )
    high = float(
        max(pooled_patient["true_patient"].max(), pooled_patient["pred_patient"].max())
    )
    ax.plot([low, high], [low, high], linestyle="--")
    ax.set_xlabel("True patient-level PMA (weeks)")
    ax.set_ylabel("Predicted patient-level PMA (weeks)")
    ax.set_title(
        f"{model_name}: 2D flattened nested OOF\n"
        f"MAE={patient_metrics['mae']:.3f}, "
        f"RMSE={patient_metrics['rmse']:.3f}, "
        f"R²={patient_metrics['r2']:.3f}"
    )
    fig.tight_layout()
    fig.savefig(
        os.path.join(EXP_ROOT, f"FINAL_patient_true_vs_pred_{model_name}.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

pooled_summary_df = pd.DataFrame(pooled_summary_rows)
pooled_summary_df.to_csv(
    os.path.join(EXP_ROOT, "FINAL_2d_baseline_summary.csv"), index=False
)
atomic_json_dump(
    {
        "experiment_hash": EXPERIMENT_HASH,
        "summary": pooled_summary_rows,
    },
    os.path.join(EXP_ROOT, "FINAL_2d_baseline_summary.json"),
)


# ============================================================
# 15. Optional paired comparison with the completed CNN OOF results
# ============================================================
if os.path.isfile(MAIN_CNN_PATIENT_OOF_CSV):
    cnn = pd.read_csv(MAIN_CNN_PATIENT_OOF_CSV, dtype={"pid": str})
    cnn_pred_col = (
        "pred_patient_mean"
        if "pred_patient_mean" in cnn.columns
        else "pred_patient"
    )
    required_cnn = {"pid", "true_patient", cnn_pred_col}
    if required_cnn.issubset(cnn.columns):
        cnn_small = cnn[["pid", "true_patient", cnn_pred_col]].copy()
        cnn_small = cnn_small.rename(columns={cnn_pred_col: "pred_patient_cnn"})
        cnn_small["pid"] = cnn_small["pid"].astype(str)
        cnn_small["abs_error_cnn"] = np.abs(
            cnn_small["pred_patient_cnn"] - cnn_small["true_patient"]
        )

        comparison = cnn_small.copy()
        for model_name in MODELS_TO_RUN:
            baseline = pd.read_csv(
                os.path.join(
                    EXP_ROOT,
                    f"FINAL_nested_oof_patient_predictions_{model_name}.csv",
                ),
                dtype={"pid": str},
            )
            baseline = baseline[["pid", "true_patient", "pred_patient", "abs_error"]]
            baseline = baseline.rename(
                columns={
                    "pred_patient": f"pred_patient_{model_name}",
                    "abs_error": f"abs_error_{model_name}",
                    "true_patient": f"true_patient_{model_name}",
                }
            )
            comparison = comparison.merge(baseline, on="pid", how="inner")
            comparison[f"abs_error_difference_{model_name}_minus_cnn"] = (
                comparison[f"abs_error_{model_name}"] - comparison["abs_error_cnn"]
            )

        comparison.to_csv(
            os.path.join(EXP_ROOT, "FINAL_paired_patient_comparison_with_CNN.csv"),
            index=False,
        )
    else:
        warnings.warn(
            "CNN patient OOF CSV exists but does not contain the expected columns; "
            "paired comparison was skipped."
        )
else:
    warnings.warn(
        "MAIN_CNN_PATIENT_OOF_CSV was not found. Baseline completed, but the "
        "optional paired CNN comparison was skipped."
    )

print("\n" + "=" * 110)
print("2D FLATTENED BASELINE NESTED CV COMPLETE")
print(pooled_summary_df.to_string(index=False))
print("Outputs saved to:", EXP_ROOT)
print("=" * 110)

In [ ]:
# ============================================================
# GA-bin post-hoc analysis for completed 2D Ridge + XGBoost
# Paste directly after the finished 2D baseline cell.
#
# Generates TWO complete versions:
#   1) ALL PMA bins, without patient-count filtering
#   2) FILTERED PMA bins, retaining bins with >= 5 patients
#
# It does NOT retrain any model.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

MIN_PATIENTS_PER_PMA_BIN = 5
PMA_BIN_WIDTH = 0.5

GA_DIR = os.path.join(EXP_ROOT, "ga_bin_analysis")
os.makedirs(GA_DIR, exist_ok=True)

MODELS = ("ridge", "xgboost")

GROUP_ORDER = [
    "EP < 26 wk",
    "VP [26,29) wk",
    "MP [29,32) wk",
    "LP [32,36) wk",
]

GROUP_LABELS = {
    "EP < 26 wk": "Extreme preterm (<26 wk)",
    "VP [26,29) wk": "Very preterm ([26,29) wk)",
    "MP [29,32) wk": "Moderate preterm ([29,32) wk)",
    "LP [32,36) wk": "Late preterm ([32,36) wk)",
}

GROUP_COLORS = {
    "EP < 26 wk": "#b24726",
    "VP [26,29) wk": "#d49a27",
    "MP [29,32) wk": "#4c9a8a",
    "LP [32,36) wk": "#4a78c2",
}


def norm_pid(value):
    text = str(value)
    if text.endswith(".0") and text[:-2].isdigit():
        return text[:-2]
    return text


def assign_ga_group(ga):
    if pd.isna(ga):
        return np.nan
    ga = float(ga)
    if ga < 26:
        return "EP < 26 wk"
    if ga < 29:
        return "VP [26,29) wk"
    if ga < 32:
        return "MP [29,32) wk"
    if ga < 36:
        return "LP [32,36) wk"
    return np.nan


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return {
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "bias": np.nan,
            "n": 0,
        }

    error = y_pred - y_true
    r2 = (
        float(r2_score(y_true, y_pred))
        if len(y_true) > 1 and not np.allclose(y_true, y_true[0])
        else np.nan
    )

    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "r2": r2,
        "bias": float(np.mean(error)),
        "n": int(len(y_true)),
    }


def make_pma_bin_table(window_df):
    parts = []

    for group in GROUP_ORDER:
        group_df = window_df[window_df["ga_group"] == group].copy()
        if group_df.empty:
            continue

        pma_min = float(group_df["true_window"].min())
        pma_max = float(group_df["true_window"].max())

        lower = np.floor(pma_min / PMA_BIN_WIDTH) * PMA_BIN_WIDTH
        upper = np.ceil(pma_max / PMA_BIN_WIDTH) * PMA_BIN_WIDTH

        edges = np.arange(
            lower,
            upper + PMA_BIN_WIDTH + 1e-9,
            PMA_BIN_WIDTH,
        )

        if len(edges) < 2:
            continue

        group_df["pma_bin"] = pd.cut(
            group_df["true_window"],
            bins=edges,
            include_lowest=True,
            right=True,
        )

        summary = (
            group_df.groupby("pma_bin", observed=True)
            .agg(
                true_mid=("true_window", "median"),
                pred_med=("pred_window_mean", "median"),
                pred_q1=(
                    "pred_window_mean",
                    lambda x: float(np.nanquantile(x, 0.25)),
                ),
                pred_q3=(
                    "pred_window_mean",
                    lambda x: float(np.nanquantile(x, 0.75)),
                ),
                mean_error=("error_window", "mean"),
                mae=("abs_error_window", "mean"),
                n_windows=("pred_window_mean", "size"),
                n_patients=("pid", "nunique"),
            )
            .reset_index()
        )

        summary["pma_bin"] = summary["pma_bin"].astype(str)
        summary["ga_group"] = group
        summary["ga_label"] = GROUP_LABELS[group]
        summary["keep_min5"] = (
            summary["n_patients"] >= MIN_PATIENTS_PER_PMA_BIN
        )
        parts.append(summary)

    if not parts:
        return pd.DataFrame()

    return pd.concat(parts, ignore_index=True)


def plot_binned_trajectory(bin_df, model, output_dir, suffix, title_suffix):
    """Save combined binned trajectory plus patient-count panel."""
    if bin_df.empty:
        print(f"Skipped empty binned trajectory: {model}, {suffix}")
        return

    plot_df = bin_df.dropna(
        subset=["true_mid", "pred_med", "pred_q1", "pred_q3"]
    ).copy()
    if plot_df.empty:
        print(f"Skipped invalid binned trajectory: {model}, {suffix}")
        return

    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=(10, 9),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08},
    )

    axis_min = float(min(plot_df["true_mid"].min(), plot_df["pred_med"].min()))
    axis_max = float(max(plot_df["true_mid"].max(), plot_df["pred_med"].max()))

    ax1.plot(
        [axis_min, axis_max],
        [axis_min, axis_max],
        linestyle="--",
        color="black",
        linewidth=1.5,
        label="Reference (prediction = PMA)",
    )

    for group in GROUP_ORDER:
        group_bins = plot_df[
            plot_df["ga_group"] == group
        ].sort_values("true_mid")

        if group_bins.empty:
            continue

        x = group_bins["true_mid"].to_numpy(dtype=float)
        pred_med = group_bins["pred_med"].to_numpy(dtype=float)
        pred_q1 = group_bins["pred_q1"].to_numpy(dtype=float)
        pred_q3 = group_bins["pred_q3"].to_numpy(dtype=float)
        n_patients = group_bins["n_patients"].to_numpy(dtype=float)

        ax1.plot(
            x,
            pred_med,
            marker="o",
            linewidth=2.4,
            color=GROUP_COLORS[group],
            label=GROUP_LABELS[group],
        )
        ax1.fill_between(
            x,
            pred_q1,
            pred_q3,
            color=GROUP_COLORS[group],
            alpha=0.12,
        )

        ax2.plot(
            x,
            n_patients,
            marker="o",
            linewidth=2.0,
            color=GROUP_COLORS[group],
            label=group,
        )

    ax1.set_ylabel("Median predicted maturational age (weeks)")
    ax1.set_title(
        f"{model}: binned median trajectory by birth-GA group\n{title_suffix}"
    )
    ax1.legend(loc="upper left", fontsize=9)
    ax1.grid(True, alpha=0.25)

    ax2.set_xlabel("Median PMA within 0.5-week bin")
    ax2.set_ylabel("Patient count")
    ax2.grid(True, alpha=0.25)

    fig.tight_layout()
    fig.savefig(
        os.path.join(
            output_dir,
            f"02_ga_bin_binned_trajectory_{suffix}.png",
        ),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def plot_four_panel(bin_df, model, output_dir, suffix, title_suffix):
    """Save one binned trajectory panel for each GA group."""
    if bin_df.empty:
        print(f"Skipped empty four-panel plot: {model}, {suffix}")
        return

    fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=False, sharey=False)
    axes = axes.flatten()
    fig.suptitle(
        f"{model}: binned trajectories by birth-GA group\n{title_suffix}",
        fontsize=15,
    )

    for ax, group in zip(axes, GROUP_ORDER):
        group_bins = bin_df[
            bin_df["ga_group"] == group
        ].sort_values("true_mid")

        ax.set_title(GROUP_LABELS[group], fontsize=12, fontweight="bold")
        ax.set_xlabel("Median PMA within bin")
        ax.set_ylabel("Median predicted age")
        ax.grid(True, alpha=0.25)

        if group_bins.empty:
            ax.text(
                0.5,
                0.5,
                "No valid bins",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            continue

        x = group_bins["true_mid"].to_numpy(dtype=float)
        pred_med = group_bins["pred_med"].to_numpy(dtype=float)
        pred_q1 = group_bins["pred_q1"].to_numpy(dtype=float)
        pred_q3 = group_bins["pred_q3"].to_numpy(dtype=float)

        axis_min = float(min(x.min(), pred_med.min()))
        axis_max = float(max(x.max(), pred_med.max()))

        ax.plot(
            [axis_min, axis_max],
            [axis_min, axis_max],
            linestyle="--",
            color="black",
            linewidth=1.2,
        )
        ax.plot(
            x,
            pred_med,
            marker="o",
            linewidth=2.4,
            color=GROUP_COLORS[group],
        )
        ax.fill_between(
            x,
            pred_q1,
            pred_q3,
            color=GROUP_COLORS[group],
            alpha=0.12,
        )

        n_bins = int(len(group_bins))
        min_patients = int(group_bins["n_patients"].min())
        max_patients = int(group_bins["n_patients"].max())
        ax.text(
            0.97,
            0.05,
            f"bins = {n_bins}\npatients/bin = {min_patients}–{max_patients}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.75),
        )

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(
        os.path.join(
            output_dir,
            f"03_ga_bin_4panel_binned_trajectory_{suffix}.png",
        ),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


# ============================================================
# 1. Extract birth GA from the static branch already in memory
# ============================================================
if "Xstatic_by_patient" not in globals() or "static_feature_cols" not in globals():
    raise RuntimeError(
        "Run this cell immediately after the completed 2D baseline cell: "
        "Xstatic_by_patient/static_feature_cols are missing."
    )

if "feats__ga_w" not in static_feature_cols:
    raise KeyError("feats__ga_w is absent from static_feature_cols.")

ga_idx = static_feature_cols.index("feats__ga_w")
pid_to_ga = {}

for pid, array in Xstatic_by_patient.items():
    array = np.asarray(array)
    if array.ndim != 2 or array.shape[1] <= ga_idx:
        continue

    ga_values = array[:, ga_idx].astype(float)
    ga_values = ga_values[np.isfinite(ga_values)]

    if len(ga_values) > 0:
        pid_to_ga[norm_pid(pid)] = float(np.median(ga_values))

print("Birth GA extracted for", len(pid_to_ga), "patients")


# ============================================================
# 2. Analyze completed Ridge and XGBoost OOF predictions
# ============================================================
all_group_tables = []
all_bin_tables = []

for model in MODELS:
    print("\n" + "=" * 100)
    print("GA-bin analysis:", model)
    print("=" * 100)

    model_dir = os.path.join(GA_DIR, model)
    os.makedirs(model_dir, exist_ok=True)

    patient_csv = os.path.join(
        EXP_ROOT,
        f"FINAL_nested_oof_patient_predictions_{model}.csv",
    )
    window_csv = os.path.join(
        EXP_ROOT,
        f"FINAL_nested_oof_window_predictions_{model}.csv",
    )

    if not os.path.isfile(patient_csv):
        raise FileNotFoundError(patient_csv)
    if not os.path.isfile(window_csv):
        raise FileNotFoundError(window_csv)

    patient_df = pd.read_csv(patient_csv, dtype={"pid": str})
    window_df = pd.read_csv(window_csv, dtype={"pid": str})

    patient_df["pid"] = patient_df["pid"].map(norm_pid)
    window_df["pid"] = window_df["pid"].map(norm_pid)

    if "pred_patient" not in patient_df.columns:
        if "pred_patient_mean" in patient_df.columns:
            patient_df = patient_df.rename(
                columns={"pred_patient_mean": "pred_patient"}
            )
        else:
            raise KeyError(f"No patient prediction column in {patient_csv}")

    if "pred_window_mean" not in window_df.columns:
        if "pred_window" in window_df.columns:
            window_df = window_df.rename(
                columns={"pred_window": "pred_window_mean"}
            )
        else:
            raise KeyError(f"No window prediction column in {window_csv}")

    for current_df in (patient_df, window_df):
        current_df["ga_birth"] = current_df["pid"].map(pid_to_ga)
        current_df["ga_group"] = current_df["ga_birth"].apply(assign_ga_group)
        current_df["model"] = model

    patient_df["error_patient"] = (
        patient_df["pred_patient"] - patient_df["true_patient"]
    )
    patient_df["abs_error_patient"] = patient_df["error_patient"].abs()

    window_df["error_window"] = (
        window_df["pred_window_mean"] - window_df["true_window"]
    )
    window_df["abs_error_window"] = window_df["error_window"].abs()

    missing_ga = sorted(
        patient_df.loc[patient_df["ga_birth"].isna(), "pid"].unique()
    )
    if missing_ga:
        pd.DataFrame({"pid": missing_ga}).to_csv(
            os.path.join(model_dir, "patients_missing_birth_ga.csv"),
            index=False,
        )
        print("Patients missing birth GA:", len(missing_ga))

    patient_df = patient_df.dropna(subset=["ga_group"]).copy()
    window_df = window_df.dropna(subset=["ga_group"]).copy()

    patient_df.to_csv(
        os.path.join(model_dir, "ga_bin_patient_predictions.csv"),
        index=False,
    )
    window_df.to_csv(
        os.path.join(model_dir, "ga_bin_window_predictions.csv"),
        index=False,
    )

    # --------------------------------------------------------
    # 2A. Metrics by birth-GA category
    # --------------------------------------------------------
    group_rows = []

    for group in GROUP_ORDER:
        patient_group = patient_df[patient_df["ga_group"] == group]
        window_group = window_df[window_df["ga_group"] == group]

        patient_metrics = regression_metrics(
            patient_group["true_patient"],
            patient_group["pred_patient"],
        )
        window_metrics = regression_metrics(
            window_group["true_window"],
            window_group["pred_window_mean"],
        )

        group_rows.append(
            {
                "model": model,
                "ga_group": group,
                "ga_label": GROUP_LABELS[group],
                "n_patients": int(patient_group["pid"].nunique()),
                "n_windows": int(len(window_group)),
                "patient_mae": patient_metrics["mae"],
                "patient_rmse": patient_metrics["rmse"],
                "patient_r2": patient_metrics["r2"],
                "patient_bias": patient_metrics["bias"],
                "window_mae": window_metrics["mae"],
                "window_rmse": window_metrics["rmse"],
                "window_r2": window_metrics["r2"],
                "window_bias": window_metrics["bias"],
            }
        )

    group_table = pd.DataFrame(group_rows)
    group_table.to_csv(
        os.path.join(model_dir, "ga_bin_group_summary.csv"),
        index=False,
    )
    all_group_tables.append(group_table)

    # --------------------------------------------------------
    # 2B. PMA-bin tables: save BOTH all and filtered versions
    # --------------------------------------------------------
    all_bins = make_pma_bin_table(window_df)
    if all_bins.empty:
        raise RuntimeError(f"No PMA bins generated for {model}.")

    all_bins["model"] = model
    filtered_bins = all_bins[all_bins["keep_min5"]].copy()

    all_bins.to_csv(
        os.path.join(model_dir, "ga_bin_pma_bins_all.csv"),
        index=False,
    )
    filtered_bins.to_csv(
        os.path.join(
            model_dir,
            f"ga_bin_pma_bins_filtered_min{MIN_PATIENTS_PER_PMA_BIN}patients.csv",
        ),
        index=False,
    )
    all_bin_tables.append(all_bins)

    # --------------------------------------------------------
    # 2C. Scatter plot using all OOF windows
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 8))

    for group in GROUP_ORDER:
        group_windows = window_df[window_df["ga_group"] == group]
        if group_windows.empty:
            continue

        ax.scatter(
            group_windows["true_window"],
            group_windows["pred_window_mean"],
            s=18,
            alpha=0.18,
            color=GROUP_COLORS[group],
            label=group,
        )

    axis_min = float(
        min(window_df["true_window"].min(), window_df["pred_window_mean"].min())
    )
    axis_max = float(
        max(window_df["true_window"].max(), window_df["pred_window_mean"].max())
    )
    reference = np.linspace(axis_min, axis_max, 300)

    ax.plot(reference, reference, linestyle="--", color="black")
    ax.fill_between(
        reference,
        reference - 1,
        reference + 1,
        color="gray",
        alpha=0.12,
    )
    ax.set_xlabel("Post-menstrual age (weeks)")
    ax.set_ylabel("Predicted maturational age (weeks)")
    ax.set_title(f"{model}: all OOF windows by birth-GA group")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        os.path.join(model_dir, "01_ga_bin_combined_scatter_all_windows.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    # --------------------------------------------------------
    # 2D. Save BOTH complete plot sets
    # --------------------------------------------------------
    plot_binned_trajectory(
        bin_df=all_bins,
        model=model,
        output_dir=model_dir,
        suffix="all",
        title_suffix="All 0.5-week PMA bins",
    )
    plot_four_panel(
        bin_df=all_bins,
        model=model,
        output_dir=model_dir,
        suffix="all",
        title_suffix="All 0.5-week PMA bins",
    )

    plot_binned_trajectory(
        bin_df=filtered_bins,
        model=model,
        output_dir=model_dir,
        suffix=f"filtered_min{MIN_PATIENTS_PER_PMA_BIN}patients",
        title_suffix=(
            f"Only bins with at least {MIN_PATIENTS_PER_PMA_BIN} patients"
        ),
    )
    plot_four_panel(
        bin_df=filtered_bins,
        model=model,
        output_dir=model_dir,
        suffix=f"filtered_min{MIN_PATIENTS_PER_PMA_BIN}patients",
        title_suffix=(
            f"Only bins with at least {MIN_PATIENTS_PER_PMA_BIN} patients"
        ),
    )

    print(
        f"Finished {model}: all bins={len(all_bins)}, "
        f"filtered bins={len(filtered_bins)}"
    )


# ============================================================
# 3. Combined outputs across Ridge and XGBoost
# ============================================================
combined_group_summary = pd.concat(all_group_tables, ignore_index=True)
combined_all_bins = pd.concat(all_bin_tables, ignore_index=True)
combined_filtered_bins = combined_all_bins[
    combined_all_bins["keep_min5"]
].copy()

combined_group_summary.to_csv(
    os.path.join(
        GA_DIR,
        "FINAL_ga_bin_group_summary_all_2d_baselines.csv",
    ),
    index=False,
)
combined_all_bins.to_csv(
    os.path.join(
        GA_DIR,
        "FINAL_ga_bin_pma_bins_all_2d_baselines.csv",
    ),
    index=False,
)
combined_filtered_bins.to_csv(
    os.path.join(
        GA_DIR,
        f"FINAL_ga_bin_pma_bins_filtered_min{MIN_PATIENTS_PER_PMA_BIN}patients.csv",
    ),
    index=False,
)

print("\n" + "=" * 100)
print("GA-BIN ANALYSIS COMPLETE")
print("Saved to:", GA_DIR)
print("Both ALL and FILTERED_MIN5 versions were generated.")
print(
    combined_group_summary[
        [
            "model",
            "ga_group",
            "n_patients",
            "patient_mae",
            "patient_rmse",
            "patient_r2",
            "patient_bias",
        ]
    ].to_string(index=False)
)
print("=" * 100)